# Experiment design

The test targets the step with the most revenue at stake and that step's weakest large segment. This notebook computes the target, the baseline, a power table, an honest CUPED estimate and the SRM check. The written plan is `docs/experiment_design.md`.

In [1]:
import sys
sys.path.insert(0, "scripts")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import analysis as a

pd.set_option("display.float_format", "{:,.4f}".format)
a.chart_style()
con = a.connect()
q = lambda sql: con.sql(sql).df()
from stats import srm_check, n_per_arm

In [2]:
target = a.experiment_target(con)
target

{'step_order': 2,
 'step': 'View item',
 'dimension': 'New vs returning',
 'segment': 'New',
 'segment_rate': 0.18470129142142597,
 'segment_share': 0.7461002376653302}

In [3]:
q(f"""SELECT dimension, segment, sessions, step_conversion FROM funnel_by_segment
      WHERE step_order = {target['step_order']} ORDER BY step_conversion""")

,dimension,segment,sessions,step_conversion
0,New vs returning,New,34268,0.1847
1,First-touch channel,Paid Search,2048,0.1860
2,First-touch channel,Organic Search,16899,0.1972
3,Device,tablet,1101,0.1984
4,First-touch channel,Direct,11510,0.1994
5,Device,mobile,19809,0.2003
6,Market,Rest of world,23406,0.2013
7,Market,United States,22100,0.2026
8,Device,desktop,29345,0.2034
9,Market,India,4749,0.2038


Session to product view is the top step: 20% of sessions view a product. New sessions are its weakest large segment and three quarters of all sessions.

The test therefore goes to new users. Randomisation is by `user_pseudo_id` at first visit. The primary metric is the share of users who view a product in their first session.

## Baseline and traffic

Weeks containing Cyber Monday, Christmas or New Year are excluded, because their traffic is not typical.

In [4]:
pop = a.experiment_population(con)
baseline, weekly = pop.view_rate.mean(), pop.new_users.mean()
order_baseline = pop.order_rate.mean()
print(f"baseline first-session product-view rate {baseline:.2%}, {weekly:,.0f} new users per week, first-session order rate {order_baseline:.2%}")
pop

baseline first-session product-view rate 17.95%, 23,349 new users per week, first-session order rate 0.60%


,week,new_users,view_rate,order_rate
0,2020-12-07,27939,0.1876,0.0092
1,2020-12-14,24451,0.1855,0.0073
2,2021-01-04,22665,0.1376,0.0032
3,2021-01-11,21238,0.1604,0.0039
4,2021-01-18,20452,0.2264,0.0065


## Power

Two-sided α = 0.05, power 0.8, 50/50 split, at least two full weeks.

In [5]:
power = a.power_table(baseline, weekly)
power

,relative_mde,absolute_mde_pts,n_per_arm,weeks_needed,months_needed
0,0.0200,0.3590,180790,16,3.7000
1,0.0300,0.5385,80660,7,1.6000
2,0.0500,0.8975,29259,3,0.7000
3,0.0750,1.3462,13126,2,0.5000
4,0.1000,1.7950,7451,2,0.5000


In [6]:
guard = pd.DataFrame([{"relative_mde": m, "n_per_arm": n_per_arm(order_baseline, m)} for m in (0.05, 0.10, 0.20)])
guard["weeks_needed"] = np.ceil(guard.n_per_arm * 2 / weekly).astype(int)
guard

,relative_mde,n_per_arm,weeks_needed
0,0.0500,1061968,91
1,0.1000,271926,24
2,0.2000,71197,7


A 5% relative lift (about 0.9 points) needs about 29,000 users per arm, three weeks of new-user traffic. That is the plan.

Order conversion cannot be powered on the same traffic. Detecting even a 10% change in first-session orders would take months, so it is a guardrail tested for harm, not a success metric.

## CUPED

CUPED needs a pre-period covariate. New users have none, so for the primary population the variance reduction is zero. The estimate below applies only if the test were widened to returning users. Covariate: viewed a product in the four weeks before; outcome: viewed a product in a two-week window.

In [7]:
cuped = a.cuped_estimate(con)
cuped

{'returning_users': 2267,
 'outcome_rate': np.float64(0.38641376268195854),
 'covariate_rate': np.float64(0.47198941332157035),
 'correlation': np.float64(0.3856843740867756),
 'theta': np.float64(0.3761912569585957),
 'variance_reduction': np.float64(0.14875243641470814)}

For returning users, CUPED would cut variance by about 15%, so it would need about 15% fewer users. That is worth using if returning users are ever added, but it changes nothing for this test.

## SRM check

Run daily from day one. A chi-square p-value below 0.001 means randomisation is broken: stop and debug rather than read the results.

In [8]:
n = int(power.loc[power.relative_mde == 0.05, "n_per_arm"].iloc[0])
examples = pd.DataFrame([
    {"case": "healthy 50/50", "control": n, "treatment": n + 150},
    {"case": "broken redirect", "control": n, "treatment": int(n * 0.97)},
])
examples[["p_value", "srm"]] = [srm_check([c, t]) for c, t in zip(examples.control, examples.treatment)]
con.close()
examples

,case,control,treatment,p_value,srm
0,healthy 50/50,29259,29409,0.5357,False
1,broken redirect,29259,28381,0.0003,True
